In [1]:
from pathlib import Path
import os
import re
import json
import gzip
import random
import unicodedata
from collections import Counter

import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader, random_split

from warcio.archiveiterator import ArchiveIterator
from bs4 import BeautifulSoup
import trafilatura

from langdetect import detect, LangDetectException
import ftfy

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

try:
    import lightning.pytorch as pl
except Exception:
    import pytorch_lightning as pl

In [2]:
PROJECT_ROOT = Path(".").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
TOKENIZER_DIR = PROJECT_ROOT / "tokenizers"

for d in [DATA_DIR, RAW_DIR, PROCESSED_DIR, TOKENIZER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
devices = [torch.cuda.device(i) for i in range(torch.cuda.device_count())]
for device in devices:
    print(torch.cuda.get_device_name(device))

NVIDIA GeForce RTX 5070 Ti


In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

### Загрузка

In [5]:
COMMON_CRAWL_COLLECTION = "CC-MAIN-2024-10"
MAX_WARC_FILES = 1

def download_file(url: str, path: Path, chunk_size: int = 1024 * 1024):
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists() and path.stat().st_size > 0:
        print(f"Файл уже существует: {path}")
        return path

    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))

        with open(path, "wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=path.name
        ) as pbar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    return path

# Функция получает список путей WARC-файлов для выбранного снапшота Common Crawl.
def get_commoncrawl_warc_paths(collection: str):
    url = f"https://data.commoncrawl.org/crawl-data/{collection}/warc.paths.gz"
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    paths = gzip.decompress(response.content).decode("utf-8").splitlines()
    return paths


warc_paths = get_commoncrawl_warc_paths(COMMON_CRAWL_COLLECTION)

print(f"Всего WARC-файлов найдено: {len(warc_paths)}")
print("Первый путь:")
print(warc_paths[0])

Всего WARC-файлов найдено: 90000
Первый путь:
crawl-data/CC-MAIN-2024-10/segments/1707947473347.0/warc/CC-MAIN-20240220211055-20240221001055-00000.warc.gz


In [6]:
downloaded_warc_files = []

for i, rel_path in enumerate(warc_paths[:MAX_WARC_FILES]):
    url = f"https://data.commoncrawl.org/{rel_path}"
    local_path = RAW_DIR / Path(rel_path).name

    print(f"Скачиваем файл {i + 1}/{MAX_WARC_FILES}")
    print(url)

    downloaded_warc_files.append(download_file(url, local_path))

downloaded_warc_files

Скачиваем файл 1/1
https://data.commoncrawl.org/crawl-data/CC-MAIN-2024-10/segments/1707947473347.0/warc/CC-MAIN-20240220211055-20240221001055-00000.warc.gz
Файл уже существует: C:\Users\521\CPM Owl\2026\Modern_neural_net_arch\HW1\data\raw\CC-MAIN-20240220211055-20240221001055-00000.warc.gz


[WindowsPath('C:/Users/521/CPM Owl/2026/Modern_neural_net_arch/HW1/data/raw/CC-MAIN-20240220211055-20240221001055-00000.warc.gz')]

### Конвертация WARC в текст

In [7]:
# ِФункция извлекает основной текст из HTML.
# Сначала используется trafilatura, затем fallback через BeautifulSoup.
def html_to_text(html: str) -> str | None:
    if not html or not html.strip():
        return None

    extracted = trafilatura.extract(
        html,
        include_comments=False,
        include_tables=False
    )

    if extracted and extracted.strip():
        return extracted.strip()

    soup = BeautifulSoup(html, "xml")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = "\n".join(line.strip() for line in text.splitlines() if line.strip())

    return text.strip() if text.strip() else None

# ِФункция конвертирует WARC в jsonl: {"url": ..., "text": ...}
def convert_warc_to_jsonl(warc_path: Path, output_path: Path, max_records: int = 2000):
    output_path.parent.mkdir(parents=True, exist_ok=True)

    saved = 0
    processed = 0

    with gzip.open(warc_path, "rb") as stream, open(output_path, "w", encoding="utf-8") as out:
        for record in tqdm(ArchiveIterator(stream), desc=f"Чтение {warc_path.name}"):
            if record.rec_type != "response":
                continue

            processed += 1

            if processed > max_records:
                break

            http_headers = record.http_headers
            if http_headers is None:
                continue

            content_type = http_headers.get_header("Content-Type") or ""

            if "text/html" not in content_type.lower():
                continue

            url = record.rec_headers.get_header("WARC-Target-URI")

            try:
                raw_bytes = record.content_stream().read()
                html = raw_bytes.decode("utf-8", errors="ignore")
            except Exception:
                continue

            text = html_to_text(html)

            if text:
                item = {
                    "url": url,
                    "text": text
                }
                out.write(json.dumps(item, ensure_ascii=False) + "\n")
                saved += 1

    print(f"Обработано WARC-записей: {processed}")
    print(f"Сохранено текстовых документов: {saved}")

    return output_path

In [8]:
cc_raw_jsonl = RAW_DIR / "commoncrawl_raw_texts.jsonl"

for warc_file in downloaded_warc_files:
    convert_warc_to_jsonl(
        warc_path=warc_file,
        output_path=cc_raw_jsonl,
        max_records=100000
    )

Чтение CC-MAIN-20240220211055-20240221001055-00000.warc.gz: 0it [00:00, ?it/s]

Обработано WARC-записей: 34998
Сохранено текстовых документов: 34132


In [9]:
def read_jsonl(path: Path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))
    return items


cc_raw_items = read_jsonl(cc_raw_jsonl)

print(f"Количество сырых документов: {len(cc_raw_items)}")
print(cc_raw_items[0]["text"][:1000])

Количество сырых документов: 34132
pGzsLw Flash PlayerAЧ Google ChromeBFirefox Microsoft Edge sAiJs][CpGz@wnϥ Flash PlayerAЬ Flash PlayerLkϥθѨMkоCU Chrome sU Firefox sU Edge s
O


### Очистка данных

Выполняется:
- удаление пустых строк;
- Unicode-нормализация;
- нормализация пробелов;
- удаление слишком коротких текстов;
- фильтрация по языку;
- разбиение слишком длинных объектов.

In [10]:
def normalize_unicode(text: str) -> str:
    text = ftfy.fix_text(text)
    text = unicodedata.normalize("NFKC", text)
    return text


def remove_control_chars(text: str) -> str:
    return "".join(
        ch for ch in text
        if ch == "\n" or ch == "\t" or not unicodedata.category(ch).startswith("C")
    )


def normalize_spaces(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]

    return "\n".join(lines).strip()


def clean_text_basic(text: str) -> str:
    text = normalize_unicode(text)
    text = remove_control_chars(text)
    text = normalize_spaces(text)
    return text


def detect_language_safe(text: str) -> str | None:
    try:
        sample = text[:2000]
        return detect(sample)
    except LangDetectException:
        return None


def is_allowed_language(text: str, allowed_langs=("en",)) -> bool:
    lang = detect_language_safe(text)
    return lang in allowed_langs


def split_long_text(
    text: str,
    max_words: int = 700,
    min_words: int = 30
):
    words = text.split()

    if len(words) < min_words:
        return []

    chunks = []

    for start in range(0, len(words), max_words):
        chunk_words = words[start:start + max_words]

        if len(chunk_words) >= min_words:
            chunks.append(" ".join(chunk_words))

    return chunks


def clean_dataset_items(
    items,
    allowed_langs=("en",),
    min_words=30,
    max_words=700
):
    cleaned = []

    for item in tqdm(items, desc="Очистка текстов"):
        text = item["text"]
        text = clean_text_basic(text)

        if not text:
            continue

        if not is_allowed_language(text, allowed_langs=allowed_langs):
            continue

        chunks = split_long_text(
            text,
            max_words=max_words,
            min_words=min_words
        )

        for chunk in chunks:
            cleaned.append({
                "url": item.get("url"),
                "text": chunk
            })

    return cleaned

In [11]:
cc_clean_items = clean_dataset_items(
    cc_raw_items,
    allowed_langs=("en",),
    min_words=30,
    max_words=700
)

print(f"После очистки документов: {len(cc_clean_items)}")
print(cc_clean_items[1]["text"][:1000])

Очистка текстов:   0%|          | 0/34132 [00:00<?, ?it/s]

После очистки документов: 19568
However it is not simply those who are traditions away option plans to help you wedding just who declare that the college has started to become outdated. Particular 42% away from worry about-demonstrated conservatives (compared with 38% of liberals and 34% out-of moderates) say a comparable- regardless of if conservatives are not likely than moderates or liberals so you're able to features ever cohabited. Also the most appropriate of one's about three ideology communities to say that this new expanding diversity during the members of the family preparations was a good crappy thing. Gender Positions; Members of the family Money Back in 1977, survey respondents were almost similarly divided anywhere between individuals who told you marriages be a little more rewarding in the event the spouse produces a living additionally the spouse protects the household and you can youngsters (43%) and those who said marriages work best whenever both partners enjoys serv

In [12]:
cc_clean_jsonl = PROCESSED_DIR / "commoncrawl_clean.jsonl"

with open(cc_clean_jsonl, "w", encoding="utf-8") as f:
    for item in cc_clean_items:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

cc_clean_jsonl

WindowsPath('C:/Users/521/CPM Owl/2026/Modern_neural_net_arch/HW1/data/processed/commoncrawl_clean.jsonl')

### Улучшение качества данных

##### Удаление дубликатов

In [13]:
def deduplicate_items(items):
    seen = set()
    unique_items = []

    for item in items:
        text = item["text"]

        if text not in seen:
            unique_items.append(item)
            seen.add(text)

    return unique_items


before = len(cc_clean_items)
cc_unique_items = deduplicate_items(cc_clean_items)
after = len(cc_unique_items)

duplicate_density = before / after if after > 0 else float("inf")

print(f"До удаления дубликатов: {before}")
print(f"После удаления дубликатов: {after}")
print(f"Дублирующая плотность данных: {duplicate_density:.4f}")

До удаления дубликатов: 19568
После удаления дубликатов: 19256
Дублирующая плотность данных: 1.0162


##### Подсчёт энтропии через GPT-2

Для каждого объекта считаем:
$$H(D) = - \sum_i \log p(D_i | D_{<i})$$
Информационная плотность:
$$\rho_{info} = \frac{H(D)}{|D|}$$

In [14]:
gpt2_tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(DEVICE)
gpt2_model.eval()

# if gpt2_tokenizer.pad_token is None:
#     gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [15]:
@torch.no_grad()
def compute_gpt2_entropy(
    text: str,
    tokenizer,
    model,
    device: str = "cpu",
    max_length: int = 1024
):
    encoded = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False
    )

    input_ids = encoded["input_ids"][0]

    if input_ids.numel() < 2:
        return {
            "entropy": np.nan,
            "num_tokens": int(input_ids.numel()),
            "info_density": np.nan
        }

    total_nll = 0.0
    total_tokens = 0

    for start in range(0, input_ids.numel(), max_length):
        chunk = input_ids[start:start + max_length]

        if chunk.numel() < 2:
            continue

        chunk = chunk.unsqueeze(0).to(device)

        outputs = model(
            input_ids=chunk,
            labels=chunk
        )

        # loss — среднее значение negative log likelihood
        # для seq_len - 1 токенов
        n_tokens = chunk.numel() - 1

        total_nll += outputs.loss.item() * n_tokens
        total_tokens += n_tokens

    if total_tokens == 0:
        return {
            "entropy": np.nan,
            "num_tokens": 0,
            "info_density": np.nan
        }

    return {
        "entropy": total_nll,
        "num_tokens": total_tokens,
        "info_density": total_nll / total_tokens
    }

In [16]:
# MAX_TEXTS_FOR_ENTROPY = min(300, len(cc_unique_items))
# cc_entropy_items = cc_unique_items[:MAX_TEXTS_FOR_ENTROPY]
cc_entropy_items = cc_unique_items

entropy_rows = []

for item in tqdm(cc_entropy_items, desc="GPT-2 entropy"):
    stats = compute_gpt2_entropy(
        item["text"],
        tokenizer=gpt2_tokenizer,
        model=gpt2_model,
        device=DEVICE
    )

    entropy_rows.append({
        "url": item.get("url"),
        "text": item["text"],
        **stats
    })

cc_entropy_df = pd.DataFrame(entropy_rows)
cc_entropy_df.head()

GPT-2 entropy:   0%|          | 0/19256 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2296 > 1024). Running this sequence through the model will result in indexing errors
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


,url,text,entropy,num_tokens,info_density
0,http://16ykm.com/vod/detail/id/854829.html,国产在线| 久久精品| 成人午夜| 无码精品| 欧美日韩一区| 热播亚洲有码 月播亚洲有码 ...,6176.371580,2293,2.693577
1,http://199cr.com/archives/25628,However it is not simply those who are traditi...,4231.639834,877,4.825131
2,http://199cr.com/archives/25628,as fifty%). Studying the show ones who've neve...,478.099666,95,5.032628
3,http://20literlife.com/manila-philippines/,"That is to say, it's a major departure from th...",2202.120604,603,3.651941
4,http://221625.tu75h.com/index.phtml?PUT=a_show...,MoMo520 | [Jڪ̷R ʁRI |M [J| Iڤ覡 ϥλ ߤ@OƦeQW| oKO...,19183.752864,4237,4.527674


In [17]:
cc_entropy_df.describe()

,entropy,num_tokens,info_density
count,19256.000000,19256.000000,19256.000000
mean,1713.931822,550.531834,3.499036
std,2288.570972,854.899643,0.926447
min,15.352834,31.000000,0.037004
25%,529.681159,142.000000,2.969798
50%,1228.993614,381.000000,3.473148
75%,2572.796790,849.000000,4.011710
max,192710.162899,51890.000000,8.093060


###### Информационная плотность всего датасета
Используем взвешенное среднее:
$$\rho_{info}(D) = \frac{\sum_i H(D_i)}{\sum_i |D_i|}$$

In [18]:
dataset_info_density = (
    cc_entropy_df["entropy"].sum()
    / cc_entropy_df["num_tokens"].sum()
)

print(f"Информационная плотность Common Crawl: {dataset_info_density:.4f}")

Информационная плотность Common Crawl: 3.1132


##### Фильтрация объектов с очень низкой и очень высокой энтропией

In [19]:
def filter_by_entropy_quantiles(
    df: pd.DataFrame,
    low_q: float = 0.05,
    high_q: float = 0.95
):
    df = df.dropna(subset=["info_density"]).copy()

    low = df["info_density"].quantile(low_q)
    high = df["info_density"].quantile(high_q)

    filtered = df[
        (df["info_density"] >= low)
        & (df["info_density"] <= high)
    ].copy()

    print(f"Нижняя граница: {low:.4f}")
    print(f"Верхняя граница: {high:.4f}")
    print(f"До фильтрации: {len(df)}")
    print(f"После фильтрации: {len(filtered)}")

    return filtered


cc_filtered_df = filter_by_entropy_quantiles(
    cc_entropy_df,
    low_q=0.05,
    high_q=0.95
)

cc_filtered_df.head()

Нижняя граница: 2.0054
Верхняя граница: 5.0871
До фильтрации: 19256
После фильтрации: 17330


,url,text,entropy,num_tokens,info_density
0,http://16ykm.com/vod/detail/id/854829.html,国产在线| 久久精品| 成人午夜| 无码精品| 欧美日韩一区| 热播亚洲有码 月播亚洲有码 ...,6176.371580,2293,2.693577
1,http://199cr.com/archives/25628,However it is not simply those who are traditi...,4231.639834,877,4.825131
2,http://199cr.com/archives/25628,as fifty%). Studying the show ones who've neve...,478.099666,95,5.032628
3,http://20literlife.com/manila-philippines/,"That is to say, it's a major departure from th...",2202.120604,603,3.651941
4,http://221625.tu75h.com/index.phtml?PUT=a_show...,MoMo520 | [Jڪ̷R ʁRI |M [J| Iڤ覡 ϥλ ߤ@OƦeQW| oKO...,19183.752864,4237,4.527674


In [20]:
cc_final_texts = cc_filtered_df["text"].tolist()

print(f"Финальное количество текстов Common Crawl: {len(cc_final_texts)}")
print(cc_final_texts[3][:1000])

Финальное количество текстов Common Crawl: 17330
That is to say, it's a major departure from the authoritative air and hard-edged beauty of Singapore – in the best possible way. The most eventful thing we did in Manila was buy our suits for Trevor's wedding. I'll get to that in a follow-up post. But there were some remarkable charms to this place, despite its rough exterior. For instance, we stayed at a great little hotel/hostel, about a quarter mile from the ocean, called the Buoy. The owner was a domineering asshole, but fortunately we only met him once. His staff, on the other hand, were a delight. The all-female staff of the attached restaurant/bar were especially sweet and great to be around, and this led us to favor the hotel for our near-constant drinking during the few days we were there. We'd sit and drink buckets of beer, and work on this stupid little website we have (maybe you've heard of it), and play pool, and hang out with the Buoy girls. We spent a lot of time that way.

### Токенизация текста

##### Символьная токенизация

In [21]:
class CharTokenizer:
    def __init__(self, special_tokens=None):
        if special_tokens is None:
            special_tokens = ["<PAD>", "<UNK>"]

        self.special_tokens = special_tokens
        self.token_to_id = {}
        self.id_to_token = {}

    def fit(self, texts):
        chars = sorted(set("".join(texts)))
        vocab = self.special_tokens + chars

        self.token_to_id = {tok: i for i, tok in enumerate(vocab)}
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}

    def encode(self, text):
        unk_id = self.token_to_id["<UNK>"]
        return [
            self.token_to_id.get(ch, unk_id)
            for ch in text
        ]

    def decode(self, ids):
        return "".join(
            self.id_to_token.get(i, "<UNK>")
            for i in ids
        )

    @property
    def vocab_size(self):
        return len(self.token_to_id)

In [22]:
char_tokenizer = CharTokenizer()
char_tokenizer.fit(cc_final_texts)

sample_text = random.choice(cc_final_texts)
char_ids = char_tokenizer.encode(sample_text)

print(f"Размер словаря символьного токенизатора: {char_tokenizer.vocab_size}")
print(f"Длина случайного объекта в символах-токенах: {len(char_ids)}")
print(sample_text[:500])

Размер словаря символьного токенизатора: 5972
Длина случайного объекта в символах-токенах: 2709
Hey Meraki Community, I have a feeling there is a simple explanation here but want to ask the gurus. Customer asks me what the 293 hidden SSIDs are showing up under the Rogue. I checked and indeed Meraki states these are recently seen on the LAN. However none of the MAC addresses show up in the Meraki client list. I am calling Meraki support and doing a packet capture now to try and figure out what these are. Any have any idea? I checked other customers and it seems this is somewhat common, and 


##### Токенизация по словам

In [23]:
class WordTokenizer:
    def __init__(self, special_tokens=None, max_vocab_size=None):
        if special_tokens is None:
            special_tokens = ["<PAD>", "<UNK>"]

        self.special_tokens = special_tokens
        self.max_vocab_size = max_vocab_size
        self.token_to_id = {}
        self.id_to_token = {}

    def _tokenize(self, text):
        return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)

    def fit(self, texts):
        counter = Counter()

        for text in texts:
            counter.update(self._tokenize(text))

        most_common = counter.most_common(self.max_vocab_size)

        vocab_words = [word for word, _ in most_common]
        vocab = self.special_tokens + vocab_words

        self.token_to_id = {tok: i for i, tok in enumerate(vocab)}
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}

    def encode(self, text):
        unk_id = self.token_to_id["<UNK>"]
        return [
            self.token_to_id.get(tok, unk_id)
            for tok in self._tokenize(text)
        ]

    def decode(self, ids):
        return " ".join(
            self.id_to_token.get(i, "<UNK>")
            for i in ids
        )

    @property
    def vocab_size(self):
        return len(self.token_to_id)

In [47]:
word_tokenizer = WordTokenizer(max_vocab_size=50000)
word_tokenizer.fit(cc_final_texts)

sample_text = random.choice(cc_final_texts)
word_ids = word_tokenizer.encode(sample_text)

print(f"Размер словаря word-токенизатора: {word_tokenizer.vocab_size}")
print(f"Длина случайного объекта в word-токенах: {len(word_ids)}")
print(sample_text[:500])

Размер словаря word-токенизатора: 50002
Длина случайного объекта в word-токенах: 94
Schuylkill County Updated 55 minutes ago * Rounding may have been employed. Prices may or may not include sales tax and can vary by location. All prices are for cash payment. Check or credit card prices may be the same or vary. Visitors are advised to check prices for themselves. BETA: The information here comes without warranty of any kind We don't know where all distributors deliver to and would recommend visitors check delivery information for themselves Delivery days to various locations may


##### BPE-токенизация

In [25]:
BPE_VOCAB_SIZE = 8000

def train_bpe_tokenizer(
    texts,
    vocab_size: int = 8000,
    save_path: Path | None = None
):
    tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
    tokenizer.decoder = ByteLevelDecoder()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
    )

    tokenizer.train_from_iterator(
        texts,
        trainer=trainer,
        length=len(texts)
    )

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        tokenizer.save(str(save_path))

    return tokenizer

In [26]:
bpe_tokenizer_path = TOKENIZER_DIR / "commoncrawl_bpe.json"

bpe_tokenizer = train_bpe_tokenizer(
    cc_final_texts,
    vocab_size=BPE_VOCAB_SIZE,
    save_path=bpe_tokenizer_path
)

sample_text = random.choice(cc_final_texts)
bpe_ids = bpe_tokenizer.encode(sample_text).ids

print(f"Размер словаря BPE-токенизатора: {bpe_tokenizer.get_vocab_size()}")
print(f"Длина случайного объекта в BPE-токенах: {len(bpe_ids)}")
print(sample_text[:500])

Размер словаря BPE-токенизатора: 8000
Длина случайного объекта в BPE-токенах: 78
6 Property Investments to Take Advantage of the Mangalore Monsoons Introduction: Property Investments in Mangalore Ever wondered why investing in property during the rainy season leaves you puzzled?... Introduction: Property Investments in Mangalore Ever wondered why investing in property during the rainy season leaves you puzzled?...


### Датасет WikiText
Теперь выполняем аналогичную обработку для wikitext.

In [27]:
wiki_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
wiki_dataset

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [28]:
wiki_raw_items = []

for split in ["train", "validation", "test"]:
    for item in wiki_dataset[split]:
        text = item["text"]

        if text and text.strip():
            wiki_raw_items.append({
                "url": f"wikitext/{split}",
                "text": text
            })

print(f"Количество сырых объектов WikiText: {len(wiki_raw_items)}")
print(wiki_raw_items[2]["text"])

Количество сырых объектов WikiText: 29119
 The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character designer Raita Honjou and composer Hitoshi Sakimoto both returned from previous entries , along with Valkyria Chronicles II director Takeshi Ozawa . A large team of writers handled the script . The game 's opening theme was sung by May 'n . 



##### Очистка WikiText

In [29]:
wiki_clean_items = clean_dataset_items(
    wiki_raw_items,
    allowed_langs=("en",),
    min_words=10,
    max_words=700
)

print(f"После очистки WikiText: {len(wiki_clean_items)}")
print(wiki_clean_items[0]["text"][:1000])

Очистка текстов:   0%|          | 0/29119 [00:00<?, ?it/s]

После очистки WikiText: 20617
Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " .


##### Удаление дубликатов WikiText

In [30]:
wiki_unique_items = deduplicate_items(wiki_clean_items)

print(f"До удаления дубликатов: {len(wiki_clean_items)}")
print(f"После удаления дубликатов: {len(wiki_unique_items)}")

До удаления дубликатов: 20617
После удаления дубликатов: 20594


##### Энтропия WikiText через GPT-2

In [31]:
# MAX_WIKI_TEXTS_FOR_ENTROPY = min(300, len(wiki_unique_items))
# wiki_entropy_items = wiki_unique_items[:MAX_WIKI_TEXTS_FOR_ENTROPY]
wiki_entropy_items = wiki_unique_items
wiki_entropy_rows = []

for item in tqdm(wiki_entropy_items, desc="WikiText GPT-2 entropy"):
    stats = compute_gpt2_entropy(
        item["text"],
        tokenizer=gpt2_tokenizer,
        model=gpt2_model,
        device=DEVICE
    )

    wiki_entropy_rows.append({
        "url": item.get("url"),
        "text": item["text"],
        **stats
    })

wiki_entropy_df = pd.DataFrame(wiki_entropy_rows)
wiki_entropy_df.head()

WikiText GPT-2 entropy:   0%|          | 0/20594 [00:00<?, ?it/s]

,url,text,entropy,num_tokens,info_density
0,wikitext/train,Senjō no Valkyria 3 : Unrecorded Chronicles ( ...,651.314315,163,3.995793
1,wikitext/train,"The game began development in 2010 , carrying ...",380.718760,104,3.660757
2,wikitext/train,"It met with positive sales in Japan , and was ...",380.628115,109,3.492001
3,wikitext/train,"As with previous Valkyira Chronicles games , V...",867.569315,228,3.805129
4,wikitext/train,"The game 's battle system , the BliTZ system ,...",1201.892713,321,3.744214


In [32]:
wiki_info_density = (
    wiki_entropy_df["entropy"].sum()
    / wiki_entropy_df["num_tokens"].sum()
)

print(f"Информационная плотность WikiText: {wiki_info_density:.4f}")

Информационная плотность WikiText: 3.8363


In [33]:
wiki_filtered_df = filter_by_entropy_quantiles(
    wiki_entropy_df,
    low_q=0.05,
    high_q=0.95
)

wiki_final_texts = wiki_filtered_df["text"].tolist()

print(f"Финальное количество текстов WikiText: {len(wiki_final_texts)}")

Нижняя граница: 3.1992
Верхняя граница: 5.5730
До фильтрации: 20594
После фильтрации: 18534
Финальное количество текстов WikiText: 18534


##### Токенизация WikiText

###### Символьная токенизация

In [34]:
wiki_char_tokenizer = CharTokenizer()
wiki_char_tokenizer.fit(wiki_final_texts)

sample_wiki_text = random.choice(wiki_final_texts)
wiki_char_ids = wiki_char_tokenizer.encode(sample_wiki_text)

print(f"WikiText char vocab size: {wiki_char_tokenizer.vocab_size}")
print(f"Длина случайного объекта: {len(wiki_char_ids)}")

WikiText char vocab size: 1036
Длина случайного объекта: 594


###### Word-токенизация

In [35]:
wiki_word_tokenizer = WordTokenizer(max_vocab_size=50000)
wiki_word_tokenizer.fit(wiki_final_texts)

sample_wiki_text = random.choice(wiki_final_texts)
wiki_word_ids = wiki_word_tokenizer.encode(sample_wiki_text)

print(f"WikiText word vocab size: {wiki_word_tokenizer.vocab_size}")
print(f"Длина случайного объекта: {len(wiki_word_ids)}")

WikiText word vocab size: 50002
Длина случайного объекта: 364


###### BPE-токенизация WikiText

Используем BPE-токенизатор, обученный на Common Crawl.

In [36]:
sample_wiki_text = random.choice(wiki_final_texts)
wiki_bpe_ids = bpe_tokenizer.encode(sample_wiki_text).ids

print(f"BPE vocab size: {bpe_tokenizer.get_vocab_size()}")
print(f"Длина случайного WikiText объекта в BPE-токенах: {len(wiki_bpe_ids)}")

BPE vocab size: 8000
Длина случайного WikiText объекта в BPE-токенах: 222


### Packed batching

Packed batching объединяет несколько коротких последовательностей в один объект фиксированной длины.

Например, если block_size = 512, то несколько коротких текстов могут быть склеены в один блок длиной 512 токенов.

Маска:
 
    0 — <PAD>;
    1 — токены первого объекта;
    2 — токены второго объекта;
    3 — токены третьего объекта и так далее.


In [37]:
PAD_TOKEN = "<PAD>"
PAD_ID = bpe_tokenizer.token_to_id(PAD_TOKEN)
PAD_ID

0

In [38]:
def encode_texts_bpe(texts, tokenizer):
    sequences = []

    for text in tqdm(texts, desc="BPE encode"):
        ids = tokenizer.encode(text).ids

        if len(ids) > 0:
            sequences.append(ids)

    return sequences


wiki_bpe_sequences = encode_texts_bpe(
    wiki_final_texts,
    bpe_tokenizer
)

print(f"Количество BPE-последовательностей WikiText: {len(wiki_bpe_sequences)}")
print(f"Пример длины: {len(wiki_bpe_sequences[0])}")

BPE encode:   0%|          | 0/18534 [00:00<?, ?it/s]

Количество BPE-последовательностей WikiText: 18534
Пример длины: 215


In [39]:
def pack_sequences(
    sequences,
    block_size: int,
    pad_id: int
):
    packed_input_ids = []
    packed_masks = []

    current_ids = []
    current_mask = []
    current_object_id = 1

    def flush():
        nonlocal current_ids, current_mask, current_object_id

        if len(current_ids) == 0:
            return

        pad_len = block_size - len(current_ids)

        current_ids = current_ids + [pad_id] * pad_len
        current_mask = current_mask + [0] * pad_len

        packed_input_ids.append(current_ids)
        packed_masks.append(current_mask)

        current_ids = []
        current_mask = []
        current_object_id = 1

    for seq in sequences:
        # Если последовательность длиннее block_size,
        # режем её на куски.
        chunks = [
            seq[i:i + block_size]
            for i in range(0, len(seq), block_size)
        ]

        for chunk in chunks:
            if len(chunk) == 0:
                continue

            # Если текущий блок переполнится, сохраняем его.
            if len(current_ids) + len(chunk) > block_size:
                flush()

            current_ids.extend(chunk)
            current_mask.extend([current_object_id] * len(chunk))
            current_object_id += 1

            # Если блок полностью заполнен, сохраняем.
            if len(current_ids) == block_size:
                flush()

    flush()

    return (
        torch.tensor(packed_input_ids, dtype=torch.long),
        torch.tensor(packed_masks, dtype=torch.long)
    )

In [40]:
BLOCK_SIZE = 512

packed_ids, packed_masks = pack_sequences(
    wiki_bpe_sequences,
    block_size=BLOCK_SIZE,
    pad_id=PAD_ID
)

print(f"Packed input ids shape: {packed_ids.shape}")
print(f"Packed masks shape: {packed_masks.shape}")

Packed input ids shape: torch.Size([8417, 512])
Packed masks shape: torch.Size([8417, 512])


In [41]:
print("Пример input_ids:")
print(packed_ids[0][:100])

print("\nПример packed mask:")
print(packed_masks[0][:100])

Пример input_ids:
tensor([  54,  220,   77,  131,  188,  736,  388, 1133,   92, 5817,  424,  998,
         625, 5224,  506,  235,  480, 1288, 2759,  299, 5960,  998, 3350,  183,
         103,  163,  207,  116, 5472, 1388,  116, 1545,   98, 1388,  108, 1545,
         208, 1388,  102, 1388,  107, 1545,   99,   22,  782,  267,  224, 1017,
         388, 1133,   92, 5817,  245,  221,  269, 3440, 3784,  424, 2591,  782,
        2090,  296, 3553,  696,  238,  341,  388, 1133,   92, 5817,  480, 1288,
        2759, 3956,   44, 3680, 3005,  782,  289,  210,  209,  448,  468, 3304,
        2076,   16,   35, 4227, 2101, 1809, 3894,  386,  706, 4012,  241, 3960,
          17,   57, 1306,  283])

Пример packed mask:
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

##### PyTorch Dataset для packed batching

In [42]:
class PackedLanguageModelingDataset(Dataset):
    def __init__(
        self,
        input_ids: torch.Tensor,
        packed_masks: torch.Tensor,
        pad_id: int
    ):
        self.input_ids = input_ids
        self.packed_masks = packed_masks
        self.pad_id = pad_id

    def __len__(self):
        return self.input_ids.shape[0]

    def __getitem__(self, idx):
        input_ids = self.input_ids[idx]
        packed_mask = self.packed_masks[idx]

        labels = input_ids.clone()

        # PAD-токены не должны участвовать в loss
        labels[input_ids == self.pad_id] = -100

        return {
            "input_ids": input_ids,
            "packed_mask": packed_mask,
            "labels": labels
        }

In [43]:
packed_dataset = PackedLanguageModelingDataset(
    input_ids=packed_ids,
    packed_masks=packed_masks,
    pad_id=PAD_ID
)

sample = packed_dataset[0]

print(sample.keys())
print(sample["input_ids"].shape)
print(sample["packed_mask"].shape)
print(sample["labels"].shape)

dict_keys(['input_ids', 'packed_mask', 'labels'])
torch.Size([512])
torch.Size([512])
torch.Size([512])


##### LightningDataModule

In [44]:
class PackedWikiTextDataModule(pl.LightningDataModule):
    def __init__(
        self,
        dataset: Dataset,
        batch_size: int = 8,
        val_ratio: float = 0.1,
        num_workers: int = 0
    ):
        super().__init__()

        self.dataset = dataset
        self.batch_size = batch_size
        self.val_ratio = val_ratio
        self.num_workers = num_workers

    def setup(self, stage=None):
        val_size = int(len(self.dataset) * self.val_ratio)
        train_size = len(self.dataset) - val_size

        self.train_dataset, self.val_dataset = random_split(
            self.dataset,
            [train_size, val_size],
            generator=torch.Generator().manual_seed(SEED)
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers
        )

In [45]:
data_module = PackedWikiTextDataModule(
    dataset=packed_dataset,
    batch_size=4,
    val_ratio=0.1,
    num_workers=0
)

data_module.setup()

batch = next(iter(data_module.train_dataloader()))

print(batch["input_ids"].shape)
print(batch["packed_mask"].shape)
print(batch["labels"].shape)

torch.Size([4, 512])
torch.Size([4, 512])
torch.Size([4, 512])


### Итоговые результаты

In [46]:
summary = {
    "Common Crawl raw documents": len(cc_raw_items),
    "Common Crawl cleaned documents": len(cc_clean_items),
    "Common Crawl unique documents": len(cc_unique_items),
    "Common Crawl final documents": len(cc_final_texts),
    "Common Crawl duplicate density": duplicate_density,
    "Common Crawl info density": dataset_info_density,
    "Char vocab size": char_tokenizer.vocab_size,
    "Word vocab size": word_tokenizer.vocab_size,
    "BPE vocab size": bpe_tokenizer.get_vocab_size(),
    "WikiText raw documents": len(wiki_raw_items),
    "WikiText cleaned documents": len(wiki_clean_items),
    "WikiText unique documents": len(wiki_unique_items),
    "WikiText final documents": len(wiki_final_texts),
    "WikiText info density": wiki_info_density,
    "Packed dataset size": len(packed_dataset),
    "Packed block size": BLOCK_SIZE
}

pd.DataFrame([summary]).T.rename(columns={0: "value"})

,value
Common Crawl raw documents,34132.000000
Common Crawl cleaned documents,19568.000000
Common Crawl unique documents,19256.000000
Common Crawl final documents,17330.000000
Common Crawl duplicate density,1.016203
Common Crawl info density,3.113229
Char vocab size,5972.000000
Word vocab size,50002.000000
BPE vocab size,8000.000000
WikiText raw documents,29119.000000


В ходе лабораторной работы были выполнены следующие пункты:
- скачан WARC-файл Common Crawl;
- HTML-страницы были извлечены из WARC и преобразованы в текст;
- выполнена первичная очистка:
  - удаление HTML;
  - удаление пустых строк;
  - нормализация Unicode;
  - нормализация пробелов;
  - фильтрация по языку;
  - разбиение длинных текстов.
- удалены дубликаты и рассчитана дублирующая плотность данных;
- с помощью GPT-2 рассчитаны:
   - энтропия объектов;
   - информационная плотность объектов;
   - информационная плотность всего датасета.
- удалены объекты с экстремально низкой и высокой информационной плотностью;
- реализованы:
   - символьная токенизация;
   - word-токенизация;
   - BPE-токенизация.
- загружен и обработан датасет WikiText;
- для WikiText выполнена токенизация с использованием BPE, обученного на Common Crawl.
- реализован механизм packed batching.
- реализован LightningDataModule для получения батчей фиксированной длины.